In [ ]:
# ==========================================
# PIPELINE FINAL DE RECOMMANDATION
# ==========================================

In [4]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering

In [6]:
import numpy as np
import pandas as pd

print(np.__version__)
print(pd.__version__)

2.4.6
3.0.5


In [8]:
# ==========================================
# RECHARGEMENT DES DONNÉES
# ==========================================

resolved_df = pd.read_excel(
    "../data/Inc_resolved by MAIT.xlsx"
)

escalated_df = pd.read_excel(
    "../data/Inc_registred by MAIT.xlsx"
)

resolved_df["source"] = "RESOLVED"
escalated_df["source"] = "ESCALATED"

print("Resolved :", resolved_df.shape)
print("Escalated :", escalated_df.shape)

C:\Users\LENOVO\PycharmProjects\DHL_AI_Incident_Assistant\.venv311\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Resolved : (1813, 12)
Escalated : (430, 12)


In [9]:
print("Colonnes RESOLVED :")
print(resolved_df.columns.tolist())

print("\nColonnes ESCALATED :")
print(escalated_df.columns.tolist())

Colonnes RESOLVED :
['Number', 'Opened', 'Target Resolution Time (GMT)', 'Informed at', 'Incident Status', 'Caller', 'Assigned Group', 'Assignee', 'Short Description', 'Description', 'Solution', 'source']

Colonnes ESCALATED :
['Number', 'Opened', 'Target Resolution Time (GMT)', 'Informed at', 'Incident Status', 'Caller', 'Assigned Group', 'Assignee', 'Short Description', 'Description', 'Solution', 'source']


In [10]:
# ==========================================
# CONSTRUCTION DE LA KNOWLEDGE BASE
# ==========================================

# Combiner les incidents résolus et escaladés
knowledge_base = pd.concat(
    [
        resolved_df,
        escalated_df
    ],
    ignore_index=True
)

# Construire le texte utilisé pour la recherche sémantique
knowledge_base["text_for_embedding"] = (
    knowledge_base["Short Description"]
    .fillna("")
    .astype(str)
    + " "
    + knowledge_base["Description"]
    .fillna("")
    .astype(str)
)

print("Knowledge base :", knowledge_base.shape)
print("Nombre d'incidents :", len(knowledge_base))

Knowledge base : (2243, 13)
Nombre d'incidents : 2243


In [11]:
print(classify_solution)

NameError: name 'classify_solution' is not defined

In [81]:
import re

def classify_solution(solution):
    if pd.isna(solution):
        return "NON_ACTIONABLE"

    s = str(solution).strip().lower()

    if not s:
        return "NON_ACTIONABLE"

    # 1. ADMINISTRATIVE
    administrative_patterns = [
        "duplicate",
        "duplicated",
        "close the ticket",
        "ticket to avoid",
        "no response from user",
        "regional ticket opened",
        "new ticket has been raised",
        "a new ticket has been raised",
        "ticket has been raised",
        "ticket opened",
        "data provided",
        "information provided",
        "wrongly routed",
        "cancelled",
        "cancelled by",
    ]

    if any(pattern in s for pattern in administrative_patterns):
        return "ADMINISTRATIVE"

    # 2. NON_ACTIONABLE
    non_actionable_patterns = [
        "issue under provider investigations",
        "provider is working on it",
        "provider is working on the issue",
        "no approval is provided",
        "required information needed to resolve",
        "required information was not provided",
        "please ask performance to provide",
        "please reach out to",
        "please contact",
        "please request access from",
        "not our responsibility",
        "no issue from our side",
        "waiting for",
        "waiting on",
        "pending",
        "under investigation",
        "changes ongoing",
        "will be changed",
        "will be configured",
        "will be installed",
    ]

    if any(pattern in s for pattern in non_actionable_patterns):
        return "NON_ACTIONABLE"

    # 3. GENERIC
    generic_solutions = {
        "resolved",
        "resolved.",
        "resolved,",
        "solved",
        "solved.",
        "done",
        "done.",
        "fixed",
        "fixed.",
        "closed",
        "closed.",
        "completed",
        "completed.",
        "granted",
        "access granted",
        "access granted.",
        "necessary done",
        "necessary done.",
        "necessaey done.",
        "necessary changes are made.",
        "the necessary changes are made.",
        "change done",
        "change done.",
        "fixed1",
    }

    if s in generic_solutions:
        return "GENERIC"

    if re.match(r"^(resolved|fixed)\s+inc[\w-]+", s):
        return "GENERIC"

    # 4. ACTIONABLE
    actionable_patterns = [
        "installed",
        "installation done",
        "created",
        "was created",
        "has been created",
        "configured",
        "configuration done",
        "updated",
        "has been updated",
        "was updated",
        "changed",
        "has been changed",
        "was changed",
        "replaced",
        "has been replaced",
        "removed",
        "has been removed",
        "reset",
        "password generated",
        "password has been sent",
        "password was sent",
        "unlocked",
        "unlock account",
        "access granted",
        "access provided",
        "granted the",
        "enabled",
        "disabled",
        "recreated",
        "fixed by",
        "released",
        "session released",
        "restored",
        "activated",
        "completed successfully",
        "request completed",
        "changes complete",
        "changes were made",
        "changes are made",
        "set as requested",
        "printer has been created",
        "printers were recreated",
        "rebooted",
        "reconfigured",
        "configured a new",
        "synced",
        "provided the access",
        "has been granted",
    ]

    if any(pattern in s for pattern in actionable_patterns):
        return "ACTIONABLE"

    # 5. Par défaut
    return "NON_ACTIONABLE"

In [82]:
knowledge_base["solution_type"] = knowledge_base["Solution"].apply(
    classify_solution
)

print(knowledge_base["solution_type"].value_counts())

solution_type
GENERIC           1401
NON_ACTIONABLE     587
ACTIONABLE         213
ADMINISTRATIVE      42
Name: count, dtype: int64


In [14]:
print(
    knowledge_base[
        ["Number", "Short Description", "Solution", "solution_type"]
    ].head(10)
)

        Number                                  Short Description  \
0  INC61070901  Free space on data disk F: is below threshold ...   
1  INC62031525  TN SPS files transmitted to DCSFTP server but ...   
2  INC67542407  MA EXP Application e-HCM OASYS not Working via...   
3  INC63115290       Update ELMs / IP Whitelisting and URL Update   
4  INC61393750  MA EXP TUNSHIP Rrovide PID range for TN local ...   
5  INC66441487                            Print issue in GCA7 app   
6  INC64928937                                      DCE BAD ISSUE   
7  INC64305092            MA YYYYY Casablanca DCVS printing issue   
8  INC61219128   MA - EXP - Probleme d'mpression et scan des docs   
9  INC64864098  MA - EXP - le montant de la NDD ne figure pas ...   

                                   Solution   solution_type  
0                        Database log fixed      ACTIONABLE  
1  Files transmitted, and SD record visible  NON_ACTIONABLE  
2                                    Closed  NON_ACTIO

In [15]:
# ==========================================
# RECHARGEMENT DES EMBEDDINGS EXISTANTS
# ==========================================

embeddings = np.load("../data/ticket_embeddings.npy")
solution_embeddings = np.load("../data/solution_embeddings.npy")

print("Ticket embeddings :", embeddings.shape)
print("Solution embeddings :", solution_embeddings.shape)
print("Knowledge base :", knowledge_base.shape)

Ticket embeddings : (2243, 384)
Solution embeddings : (2243, 384)
Knowledge base : (2243, 14)


In [75]:
# ==========================================
# CLASSIFICATION DES SOLUTIONS
# ==========================================

def classify_solution(solution):

    solution = str(solution).strip().lower()

    # --------------------------------------
    # 1. Solution vide
    # --------------------------------------

    if not solution or solution == "nan":
        return "NON_ACTIONABLE"

    # --------------------------------------
    # 2. Réponses administratives
    # --------------------------------------

    administrative_keywords = [
        "duplicate",
        "duplicated",
        "close the ticket",
        "ticket to avoid",
        "no response from user",
        "regional ticket opened",
        "new ticket has been raised",
        "a new ticket has been raised",
        "data provided",
        "cancelled",
        "cancelled by"
    ]

    for keyword in administrative_keywords:
        if keyword in solution:
            return "ADMINISTRATIVE"

    # --------------------------------------
    # 3. Réponses génériques
    # --------------------------------------

    generic_solutions = {
        "resolved",
        "resolved.",
        "resolved,",
        "solved",
        "solved.",
        "done",
        "done.",
        "fixed",
        "fixed.",
        "closed",
        "closed.",
        "granted",
        "access granted",
        "access granted.",
        "necessary done",
        "necessary done.",
        "necessaey done.",
        "issue fixed",
        "issue solved",
        "problem solved"
    }

    if solution in generic_solutions:
        return "GENERIC"

    if solution.startswith("resolved inc"):
        return "GENERIC"

    if solution.startswith("fixed inc"):
        return "GENERIC"

    # --------------------------------------
    # 4. Actions techniques concrètes
    # --------------------------------------

    actionable_keywords = [
        "configured",
        "configuration",
        "setup",
        "set up",
        "installed",
        "installation",
        "reinstalled",
        "restart",
        "restarted",
        "reset",
        "repaired",
        "updated",
        "update",
        "changed",
        "replaced",
        "enabled",
        "disabled",
        "created",
        "removed",
        "deleted",
        "password reset",
        "driver",
        "manually",
        "performed",
        "applied"
    ]

    for keyword in actionable_keywords:
        if keyword in solution:
            return "ACTIONABLE"

    # --------------------------------------
    # 5. Par défaut
    # --------------------------------------

    return "NON_ACTIONABLE"


knowledge_base["solution_type"] = (
    knowledge_base["Solution"]
    .apply(classify_solution)
)

print(
    knowledge_base["solution_type"]
    .value_counts()
)

solution_type
GENERIC           1393
NON_ACTIONABLE     636
ACTIONABLE         178
ADMINISTRATIVE      36
Name: count, dtype: int64


In [17]:
# ==========================================
# 1. RECHERCHE DES TICKETS SIMILAIRES
# ==========================================

def search_similar_tickets(query, top_k=20):
    """
    Recherche les tickets historiques les plus similaires
    à un nouveau ticket.
    """

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    # Normalisation des embeddings historiques
    ticket_embeddings_norm = embeddings / np.linalg.norm(
        embeddings,
        axis=1,
        keepdims=True
    )

    # Similarité cosinus
    similarities = ticket_embeddings_norm @ query_embedding

    # Top-K
    top_indices = np.argsort(similarities)[::-1][:top_k]

    candidates = knowledge_base.iloc[top_indices].copy()

    candidates["Ticket_Similarity"] = similarities[top_indices]

    return candidates.sort_values(
        "Ticket_Similarity",
        ascending=False
    ).reset_index(drop=True)

In [84]:
# ==========================================
# DÉTECTION DES SOLUTIONS NON INFORMATIVES
# ==========================================

def is_non_informative_solution(solution):

    if pd.isna(solution):
        return True

    text = str(solution).strip().lower()

    if not text:
        return True

    non_informative = {
        "resolved",
        "resolved.",
        "resolve",
        "solved",
        "solved.",
        "closed",
        "closed.",
        "done",
        "done.",
        "fixed",
        "fixed."
    }

    if text in non_informative:
        return True

    automatic_patterns = [
        "rule of 3",
        "rule of three",
        "set to status 'resolved'",
        "set to status resolved",
        "ticket was set to status"
    ]

    return any(
        pattern in text
        for pattern in automatic_patterns
    )

In [85]:
knowledge_base["non_informative"] = (
    knowledge_base["Solution"]
    .apply(is_non_informative_solution)
)

In [83]:
# ==========================================
# FILTRAGE DES SOLUTIONS UTILISABLES
# ==========================================

def filter_actionable_candidates(candidates):
    """
    Conserve les solutions réellement exploitables.

    Priorité :
    1. ACTIONABLE
    2. Si aucune solution ACTIONABLE :
       utiliser les solutions GENERIC qui
       ne sont pas purement non-informatives.
    """

    candidates = candidates.copy()

    # --------------------------------------
    # 1. Retirer les solutions non informatives
    # --------------------------------------

    if "non_informative" in candidates.columns:

        usable = candidates[
            ~candidates["non_informative"]
        ].copy()

    else:

        usable = candidates.copy()

    # --------------------------------------
    # 2. Priorité aux ACTIONABLE
    # --------------------------------------

    actionable = usable[
        usable["solution_type"] == "ACTIONABLE"
    ].copy()

    if not actionable.empty:
        return actionable

    # --------------------------------------
    # 3. Fallback GENERIC
    # --------------------------------------

    generic = usable[
        usable["solution_type"] == "GENERIC"
    ].copy()

    return generic

In [21]:
query = """
GCA printer is not working and delivery sheets cannot be printed.
"""

candidates = search_similar_tickets(
    query,
    top_k=20
)

actionable = filter_actionable_candidates(candidates)

print("Tickets récupérés :", len(candidates))
print("Tickets ACTIONABLE :", len(actionable))

display(
    actionable[
        [
            "Number",
            "Short Description",
            "Solution",
            "Ticket_Similarity",
            "solution_type"
        ]
    ]
)

Tickets récupérés : 20
Tickets ACTIONABLE : 10


,Number,Short Description,Solution,Ticket_Similarity,solution_type
0,INC66310792,Issue printing Delivery Sheet from GCA - (GCA ...,Printer has been fixed.\nResolved.,0.780634,ACTIONABLE
1,INC65363499,Issue printing from GCA after printer model ch...,Printer setup has been updated and it is worki...,0.663335,ACTIONABLE
4,INC66441487,Print issue in GCA7 app,TN Provider is working on it,0.645664,ACTIONABLE
5,INC64341450,GCA 7 (Delivery sheet) printing delay issue,Printer drivers were updated and issue resolved.,0.626882,ACTIONABLE
6,INC65033159,GCA 7 (Delivery sheet) printing delay issue,Mentioned printers were recreated on both serv...,0.623463,ACTIONABLE
8,INC64304524,RW printer not print from DCVS,"Dear colleague, because the required informati...",0.585298,ACTIONABLE
9,INC65410555,Request to change printer model for GCA printe...,As per the request printer driver updated \n\n...,0.583644,ACTIONABLE
10,INC61302569,Add printers on GCA7 app,"Completed, printers installed on demanded serv...",0.583277,ACTIONABLE
14,INC61933011,Printer not responding,"Hello,\nWe found that source IPs are not added...",0.574992,ACTIONABLE
15,INC65461130,Request to change printer model for GCA printe...,Get sys admin to update the driver based on th...,0.565794,ACTIONABLE


In [20]:
# ==========================================
# RESTAURATION DU MODÈLE
# ==========================================

from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

print("Modèle chargé :", model)

C:\Users\LENOVO\PycharmProjects\DHL_AI_Incident_Assistant\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2303.41it/s]


Modèle chargé : SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)


In [22]:
# ==========================================
# 3. CLUSTERING DES SOLUTIONS
# ==========================================

def cluster_similar_solutions(
    solution_similarity_matrix,
    threshold=0.75
):
    """
    Regroupe les solutions historiquement similaires.

    threshold = seuil minimal de similarité entre solutions.
    """

    distance_matrix = 1 - solution_similarity_matrix.values

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=1 - threshold
    )

    labels = clustering.fit_predict(distance_matrix)

    clusters = pd.DataFrame({
        "Number": solution_similarity_matrix.index,
        "Cluster": labels
    })

    return clusters

In [23]:
# ==========================================
# 4. EMBEDDINGS DES SOLUTIONS CANDIDATES
# ==========================================

solution_texts = (
    actionable["Solution"]
    .fillna("")
    .astype(str)
    .tolist()
)

candidate_solution_embeddings = model.encode(
    solution_texts,
    normalize_embeddings=True,
    show_progress_bar=False
)

solution_matrix = cosine_similarity(
    candidate_solution_embeddings
)

solution_similarity_matrix = pd.DataFrame(
    solution_matrix,
    index=actionable["Number"],
    columns=actionable["Number"]
)

print(
    "Matrice de similarité des solutions :",
    solution_similarity_matrix.shape
)

Matrice de similarité des solutions : (10, 10)


In [24]:
clusters = cluster_similar_solutions(
    solution_similarity_matrix,
    threshold=0.75
)

display(
    clusters.sort_values("Cluster")
)

,Number,Cluster
7,INC61302569,0
4,INC65033159,0
9,INC65461130,1
0,INC66310792,2
3,INC64341450,2
1,INC65363499,2
2,INC66441487,3
8,INC61933011,4
5,INC64304524,5
6,INC65410555,6


In [25]:
# ==========================================
# 5. SCORING DES CLUSTERS
# ==========================================

def score_solution_clusters(
    actionable,
    clusters,
    solution_similarity_matrix
):
    """
    Calcule un score pour chaque cluster de solutions.

    Le score prend en compte :
    - pertinence des tickets
    - cohérence des solutions
    - nombre de tickets dans le cluster
    """

    cluster_results = []

    for cluster_id in sorted(clusters["Cluster"].unique()):

        cluster_numbers = clusters.loc[
            clusters["Cluster"] == cluster_id,
            "Number"
        ].tolist()

        cluster_data = actionable[
            actionable["Number"].isin(cluster_numbers)
        ].copy()

        if cluster_data.empty:
            continue

        # Pertinence moyenne des tickets
        ticket_similarity_mean = (
            cluster_data["Ticket_Similarity"].mean()
        )

        # Meilleure pertinence du cluster
        ticket_similarity_max = (
            cluster_data["Ticket_Similarity"].max()
        )

        # Cohérence entre les solutions
        valid_tickets = cluster_data["Number"].tolist()

        if len(valid_tickets) > 1:

            matrix = solution_similarity_matrix.loc[
                valid_tickets,
                valid_tickets
            ].values

            upper_values = matrix[
                np.triu_indices(
                    len(matrix),
                    k=1
                )
            ]

            solution_coherence = upper_values.mean()

        else:
            # Une seule solution = pas de conflit interne
            solution_coherence = 1.0

        cluster_results.append({
            "Cluster": cluster_id,
            "Number_of_Tickets": len(cluster_data),
            "Ticket_Similarity_Mean": ticket_similarity_mean,
            "Ticket_Similarity_Max": ticket_similarity_max,
            "Solution_Coherence": solution_coherence
        })

    return pd.DataFrame(cluster_results)

In [26]:
# ==========================================
# CALCUL DES SCORES
# ==========================================

cluster_scores = score_solution_clusters(
    actionable,
    clusters,
    solution_similarity_matrix
)

display(cluster_scores)

,Cluster,Number_of_Tickets,Ticket_Similarity_Mean,Ticket_Similarity_Max,Solution_Coherence
0,0,2,0.603370,0.623463,0.807221
1,1,1,0.565794,0.565794,1.000000
2,2,3,0.690284,0.780634,0.826594
3,3,1,0.645664,0.645664,1.000000
4,4,1,0.574992,0.574992,1.000000
5,5,1,0.585298,0.585298,1.000000
6,6,1,0.583644,0.583644,1.000000


In [27]:
# ==========================================
# 6. CLASSEMENT DES CLUSTERS
# ==========================================

def rank_solution_clusters(cluster_scores):

    if cluster_scores.empty:
        return cluster_scores.copy()

    scores = cluster_scores.copy()

    # Nombre maximal de tickets dans un cluster
    max_support = scores["Number_of_Tickets"].max()

    # Pertinence
    scores["Relevance_Score"] = (
        scores["Ticket_Similarity_Mean"]
    )

    # Cohérence des solutions
    scores["Coherence_Score"] = (
        scores["Solution_Coherence"]
    )

    # Support historique
    scores["Support_Score"] = (
        scores["Number_of_Tickets"] / max_support
    )

    # Score final
    scores["Cluster_Final_Score"] = (
        0.5 * scores["Relevance_Score"]
        + 0.3 * scores["Coherence_Score"]
        + 0.2 * scores["Support_Score"]
    )

    return scores.sort_values(
        "Cluster_Final_Score",
        ascending=False
    ).reset_index(drop=True)

In [28]:
ranked_clusters = rank_solution_clusters(
    cluster_scores
)

display(ranked_clusters)

,Cluster,Number_of_Tickets,Ticket_Similarity_Mean,Ticket_Similarity_Max,Solution_Coherence,Relevance_Score,Coherence_Score,Support_Score,Cluster_Final_Score
0,2,3,0.690284,0.780634,0.826594,0.690284,0.826594,1.000000,0.793120
1,3,1,0.645664,0.645664,1.000000,0.645664,1.000000,0.333333,0.689499
2,0,2,0.603370,0.623463,0.807221,0.603370,0.807221,0.666667,0.677184
3,5,1,0.585298,0.585298,1.000000,0.585298,1.000000,0.333333,0.659316
4,6,1,0.583644,0.583644,1.000000,0.583644,1.000000,0.333333,0.658489
5,4,1,0.574992,0.574992,1.000000,0.574992,1.000000,0.333333,0.654163
6,1,1,0.565794,0.565794,1.000000,0.565794,1.000000,0.333333,0.649564


In [29]:
# ==========================================
# 7. SÉLECTION DE LA SOLUTION REPRÉSENTATIVE
# ==========================================

def select_cluster_representatives(
    actionable,
    clusters,
    solution_similarity_matrix
):
    """
    Sélectionne une solution représentative
    pour chaque cluster.

    Pour un cluster contenant plusieurs solutions,
    la solution représentative est celle ayant la
    similarité moyenne la plus élevée avec les
    autres solutions du cluster.
    """

    representatives = []

    # Parcourir chaque cluster
    for cluster_id in sorted(
        clusters["Cluster"].unique()
    ):

        # Tickets appartenant au cluster
        cluster_numbers = clusters.loc[
            clusters["Cluster"] == cluster_id,
            "Number"
        ].tolist()

        # Données correspondantes
        cluster_data = actionable[
            actionable["Number"].isin(
                cluster_numbers
            )
        ].copy()

        if cluster_data.empty:
            continue

        # --------------------------------------
        # CAS 1 : Cluster avec une seule solution
        # --------------------------------------

        if len(cluster_numbers) == 1:

            row = cluster_data.iloc[0]

            representatives.append({

                "Cluster": cluster_id,

                "Representative_Ticket":
                    row["Number"],

                "Representative_Solution":
                    row["Solution"],

                "Ticket_Similarity":
                    row["Ticket_Similarity"],

                "Internal_Similarity":
                    1.0
            })

        # --------------------------------------
        # CAS 2 : Plusieurs solutions
        # --------------------------------------

        else:

            # Sous-matrice du cluster
            cluster_matrix = (
                solution_similarity_matrix.loc[
                    cluster_numbers,
                    cluster_numbers
                ]
            )

            mean_similarities = {}

            # Calculer la similarité moyenne
            # de chaque solution avec les autres
            for ticket_number in cluster_numbers:

                other_tickets = [

                    t
                    for t in cluster_numbers

                    if t != ticket_number

                ]

                mean_similarity = (
                    cluster_matrix.loc[
                        ticket_number,
                        other_tickets
                    ].mean()
                )

                mean_similarities[
                    ticket_number
                ] = mean_similarity

            # Ticket le plus central
            representative_ticket = max(
                mean_similarities,
                key=mean_similarities.get
            )

            representative_row = cluster_data[
                cluster_data["Number"]
                == representative_ticket
            ].iloc[0]

            representatives.append({

                "Cluster": cluster_id,

                "Representative_Ticket":
                    representative_ticket,

                "Representative_Solution":
                    representative_row["Solution"],

                "Ticket_Similarity":
                    representative_row[
                        "Ticket_Similarity"
                    ],

                "Internal_Similarity":
                    mean_similarities[
                        representative_ticket
                    ]
            })

    return pd.DataFrame(
        representatives
    )

In [30]:
# ==========================================
# CALCUL DES SOLUTIONS REPRÉSENTATIVES
# ==========================================

representatives = (
    select_cluster_representatives(
        actionable,
        clusters,
        solution_similarity_matrix
    )
)

display(
    representatives.sort_values(
        "Cluster"
    )
)

,Cluster,Representative_Ticket,Representative_Solution,Ticket_Similarity,Internal_Similarity
0,0,INC65033159,Mentioned printers were recreated on both serv...,0.623463,0.807221
1,1,INC65461130,Get sys admin to update the driver based on th...,0.565794,1.000000
2,2,INC65363499,Printer setup has been updated and it is worki...,0.663335,0.828543
3,3,INC66441487,TN Provider is working on it,0.645664,1.000000
4,4,INC61933011,"Hello,\nWe found that source IPs are not added...",0.574992,1.000000
5,5,INC64304524,"Dear colleague, because the required informati...",0.585298,1.000000
6,6,INC65410555,As per the request printer driver updated \n\n...,0.583644,1.000000


In [31]:
# ==========================================
# 8. SÉLECTION DU CLUSTER GAGNANT
# ==========================================

def select_best_cluster(
    ranked_clusters,
    representatives
):
    """
    Sélectionne le cluster ayant le meilleur score
    et récupère sa solution représentative.
    """

    if ranked_clusters.empty:
        return None

    # --------------------------------------
    # 1. Meilleur cluster
    # --------------------------------------

    best_cluster = ranked_clusters.iloc[0]

    best_cluster_id = best_cluster["Cluster"]

    # --------------------------------------
    # 2. Solution représentative associée
    # --------------------------------------

    representative = representatives[
        representatives["Cluster"]
        == best_cluster_id
    ]

    if representative.empty:
        return None

    representative = representative.iloc[0]

    # --------------------------------------
    # 3. Résultat final
    # --------------------------------------

    return {

        "Cluster":
            int(best_cluster_id),

        "Cluster_Final_Score":
            best_cluster["Cluster_Final_Score"],

        "Representative_Ticket":
            representative["Representative_Ticket"],

        "Representative_Solution":
            representative["Representative_Solution"],

        "Ticket_Similarity":
            representative["Ticket_Similarity"],

        "Internal_Similarity":
            representative["Internal_Similarity"],

        "Number_of_Tickets":
            best_cluster["Number_of_Tickets"],

        "Ticket_Similarity_Mean":
            best_cluster["Ticket_Similarity_Mean"],

        "Solution_Coherence":
            best_cluster["Solution_Coherence"]
    }

In [32]:
# ==========================================
# RÉSULTAT : CLUSTER GAGNANT
# ==========================================

best_result = select_best_cluster(
    ranked_clusters,
    representatives
)

best_result

{'Cluster': 2,
 'Cluster_Final_Score': np.float64(0.7931201040744782),
 'Representative_Ticket': 'INC65363499',
 'Representative_Solution': 'Printer setup has been updated and it is working now.\n\nResolved.\n',
 'Ticket_Similarity': np.float32(0.6633347),
 'Internal_Similarity': np.float64(0.828542947769165),
 'Number_of_Tickets': np.float64(3.0),
 'Ticket_Similarity_Mean': np.float64(0.6902837753295898),
 'Solution_Coherence': np.float64(0.8265940546989441)}

In [33]:
# ==========================================
# 9. CONSTRUCTION DES ÉVIDENCES
# ==========================================

def build_recommendation_evidence(
    query,
    best_result,
    ranked_clusters,
    clusters,
    actionable,
    top_n_tickets=5
):
    """
    Construit les informations permettant
    d'expliquer la recommandation.

    Ces informations serviront ensuite pour :
    - l'affichage
    - l'explication de la recommandation
    - la génération en langage naturel
    """

    # --------------------------------------
    # 1. Informations du cluster gagnant
    # --------------------------------------

    best_cluster_id = best_result["Cluster"]

    # --------------------------------------
    # 2. Tickets appartenant au cluster
    # --------------------------------------

    cluster_numbers = clusters.loc[
        clusters["Cluster"] == best_cluster_id,
        "Number"
    ].tolist()

    # --------------------------------------
    # 3. Données des tickets du cluster
    # --------------------------------------

    cluster_tickets = actionable[
        actionable["Number"].isin(
            cluster_numbers
        )
    ].copy()

    # Trier par pertinence
    cluster_tickets = cluster_tickets.sort_values(
        "Ticket_Similarity",
        ascending=False
    )

    # --------------------------------------
    # 4. Top tickets utilisés comme évidence
    # --------------------------------------

    evidence_tickets = cluster_tickets.head(
        top_n_tickets
    )

    evidence_tickets = evidence_tickets[
        [
            "Number",
            "Short Description",
            "Ticket_Similarity"
        ]
    ].to_dict(
        orient="records"
    )

    # --------------------------------------
    # 5. Construction de l'évidence
    # --------------------------------------

    evidence = {

        "Query":
            query,

        "Winning_Cluster":
            best_cluster_id,

        "Cluster_Score":
            float(
                best_result[
                    "Cluster_Final_Score"
                ]
            ),

        "Number_of_Tickets":
            int(
                best_result[
                    "Number_of_Tickets"
                ]
            ),

        "Ticket_Similarity_Mean":
            float(
                best_result[
                    "Ticket_Similarity_Mean"
                ]
            ),

        "Solution_Coherence":
            float(
                best_result[
                    "Solution_Coherence"
                ]
            ),

        "Representative_Ticket":
            best_result[
                "Representative_Ticket"
            ],

        "Representative_Solution":
            best_result[
                "Representative_Solution"
            ],

        "Evidence_Tickets":
            evidence_tickets
    }

    return evidence

In [34]:
# ==========================================
# TEST DES ÉVIDENCES
# ==========================================

evidence = build_recommendation_evidence(
    query=query,
    best_result=best_result,
    ranked_clusters=ranked_clusters,
    clusters=clusters,
    actionable=actionable
)

evidence

{'Query': '\nGCA printer is not working and delivery sheets cannot be printed.\n',
 'Winning_Cluster': 2,
 'Cluster_Score': 0.7931201040744782,
 'Number_of_Tickets': 3,
 'Ticket_Similarity_Mean': 0.6902837753295898,
 'Solution_Coherence': 0.8265940546989441,
 'Representative_Ticket': 'INC65363499',
 'Representative_Solution': 'Printer setup has been updated and it is working now.\n\nResolved.\n',
 'Evidence_Tickets': [{'Number': 'INC66310792',
   'Short Description': 'Issue printing Delivery Sheet from GCA - (GCA Printer :  maagapr1)',
   'Ticket_Similarity': 0.7806342244148254},
  {'Number': 'INC65363499',
   'Short Description': 'Issue printing from GCA after printer model changed !',
   'Ticket_Similarity': 0.6633347272872925},
  {'Number': 'INC64341450',
   'Short Description': 'GCA 7 (Delivery sheet) printing delay issue',
   'Ticket_Similarity': 0.6268823146820068}]}

In [35]:
# ==========================================
# 10. PIPELINE COMPLÈTE DE RECOMMANDATION
# ==========================================

def recommend_solution(
    query,
    top_k=20,
    cluster_threshold=0.75
):
    """
    Pipeline complète de recommandation.

    Étapes :

    1. Recherche des tickets similaires
    2. Filtrage des solutions ACTIONABLE
    3. Embeddings des solutions candidates
    4. Similarité entre solutions
    5. Clustering des solutions
    6. Scoring des clusters
    7. Classement des clusters
    8. Sélection des représentants
    9. Sélection du cluster gagnant
    10. Construction des évidences

    Retourne un dictionnaire contenant
    la recommandation et les informations
    permettant de l'expliquer.
    """

    # ======================================
    # 1. RECHERCHE DES TICKETS SIMILAIRES
    # ======================================

    candidates = search_similar_tickets(
        query,
        top_k=top_k
    )

    if candidates.empty:

        return {
            "status": "NO_RESULT",
            "message": (
                "Aucun ticket similaire "
                "n'a été trouvé."
            )
        }

    # ======================================
    # 2. FILTRAGE ACTIONABLE
    # ======================================

    actionable = (
        filter_actionable_candidates(
            candidates
        )
    )

    if actionable.empty:

        return {
            "status": "NO_ACTIONABLE_RESULT",
            "message": (
                "Des tickets similaires ont été "
                "trouvés, mais aucune solution "
                "ACTIONABLE n'est disponible."
            ),
            "Candidates": candidates
        }

    # ======================================
    # 3. EMBEDDINGS DES SOLUTIONS
    # ======================================

    solution_texts = (
        actionable["Solution"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    candidate_solution_embeddings = (
        model.encode(
            solution_texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )
    )

    # ======================================
    # 4. MATRICE DE SIMILARITÉ
    # ======================================

    solution_matrix = cosine_similarity(
        candidate_solution_embeddings
    )

    solution_similarity_matrix = (
        pd.DataFrame(
            solution_matrix,
            index=actionable["Number"],
            columns=actionable["Number"]
        )
    )

    # ======================================
    # 5. CLUSTERING
    # ======================================

    clusters = cluster_similar_solutions(
        solution_similarity_matrix,
        threshold=cluster_threshold
    )

    # ======================================
    # 6. SCORING DES CLUSTERS
    # ======================================

    cluster_scores = (
        score_solution_clusters(
            actionable,
            clusters,
            solution_similarity_matrix
        )
    )

    if cluster_scores.empty:

        return {
            "status": "NO_CLUSTER",
            "message": (
                "Impossible de construire "
                "des clusters de solutions."
            )
        }

    # ======================================
    # 7. CLASSEMENT DES CLUSTERS
    # ======================================

    ranked_clusters = (
        rank_solution_clusters(
            cluster_scores
        )
    )

    # ======================================
    # 8. SOLUTIONS REPRÉSENTATIVES
    # ======================================

    representatives = (
        select_cluster_representatives(
            actionable,
            clusters,
            solution_similarity_matrix
        )
    )

    # ======================================
    # 9. CLUSTER GAGNANT
    # ======================================

    best_result = select_best_cluster(
        ranked_clusters,
        representatives
    )

    if best_result is None:

        return {
            "status": "NO_RECOMMENDATION",
            "message": (
                "Impossible de sélectionner "
                "une recommandation."
            )
        }

    # ======================================
    # 10. ÉVIDENCES
    # ======================================

    evidence = (
        build_recommendation_evidence(
            query=query,
            best_result=best_result,
            ranked_clusters=ranked_clusters,
            clusters=clusters,
            actionable=actionable
        )
    )

    # ======================================
    # 11. RÉSULTAT FINAL
    # ======================================

    return {

        "status":
            "SUCCESS",

        "recommendation":
            best_result[
                "Representative_Solution"
            ],

        "evidence":
            evidence,

        # Informations supplémentaires
        # utiles pour le diagnostic

        "candidates":
            candidates,

        "actionable":
            actionable,

        "clusters":
            clusters,

        "cluster_scores":
            cluster_scores,

        "ranked_clusters":
            ranked_clusters,

        "representatives":
            representatives
    }

In [36]:
# ==========================================
# TEST DE LA PIPELINE COMPLÈTE
# ==========================================

query = """
GCA printer is not working and delivery sheets cannot be printed.
"""

result = recommend_solution(
    query,
    top_k=20,
    cluster_threshold=0.75
)

print("STATUS :")
print(result["status"])

print("\n" + "=" * 60)
print("RECOMMANDATION")
print("=" * 60)

print(
    result["recommendation"]
)

print("\n" + "=" * 60)
print("ÉVIDENCES")
print("=" * 60)

print(result["evidence"])

STATUS :
SUCCESS

RECOMMANDATION
Printer setup has been updated and it is working now.

Resolved.


ÉVIDENCES
{'Query': '\nGCA printer is not working and delivery sheets cannot be printed.\n', 'Winning_Cluster': 2, 'Cluster_Score': 0.7931201040744782, 'Number_of_Tickets': 3, 'Ticket_Similarity_Mean': 0.6902837753295898, 'Solution_Coherence': 0.8265940546989441, 'Representative_Ticket': 'INC65363499', 'Representative_Solution': 'Printer setup has been updated and it is working now.\n\nResolved.\n', 'Evidence_Tickets': [{'Number': 'INC66310792', 'Short Description': 'Issue printing Delivery Sheet from GCA - (GCA Printer :  maagapr1)', 'Ticket_Similarity': 0.7806342244148254}, {'Number': 'INC65363499', 'Short Description': 'Issue printing from GCA after printer model changed !', 'Ticket_Similarity': 0.6633347272872925}, {'Number': 'INC64341450', 'Short Description': 'GCA 7 (Delivery sheet) printing delay issue', 'Ticket_Similarity': 0.6268823146820068}]}


In [37]:
# ==========================================
# EXEMPLES DE TICKETS POUR LES TESTS
# ==========================================

display(
    knowledge_base[
        [
            "Number",
            "Short Description",
            "Description",
            "Solution",
            "solution_type"
        ]
    ]
    .sample(5, random_state=42)
)

,Number,Short Description,Description,Solution,solution_type
727,INC71877391,MA - EXP - After several attempts to access PO...,Application Name: POS\nLogin Name: rzougmi\nCo...,RESOLVED,GENERIC
907,INC69304265,MA - EXP - esecure f.jouki@enosis.ma pour le ...,New account / modification / removal: \nNew Ac...,Done.,GENERIC
1547,INC66765753,MA - EXP - Printer issue,"Dear Team , \niI inform you that i have issue ...",Resolved,GENERIC
240,INC72044971,MA - EXP - Correction of Phone Number on Invoi...,"Dear Nabil, Chetan\n\nHope you are doing well,...",Fixed by Chetan,ACTIONABLE
686,INC67571325,MA - EXP - RESET PASSWORD LDAP : zelhadda,New account / modification / removal: \nModifi...,"Hello Oussama,\n\nPlease note the new password...",ACTIONABLE


In [38]:
query_2 = """ MA - EXP - RESET PASSWORD LDAP : zelhadda New account / modification / removal: \nModifi... """

result_2 = recommend_solution(
    query_2,
    top_k=20,
    cluster_threshold=0.75
)

print("STATUS :", result_2["status"])

if result_2["status"] == "SUCCESS":

    print("\nRECOMMANDATION :")
    print(result_2["recommendation"])

    print("\nÉVIDENCES :")
    print(result_2["evidence"])

STATUS : SUCCESS

RECOMMANDATION :
Hello Oussama,

Please note the new password : Focus@147258369@

ÉVIDENCES :
{'Query': ' MA - EXP - RESET PASSWORD LDAP : zelhadda New account / modification / removal: \nModifi... ', 'Winning_Cluster': 4, 'Cluster_Score': 0.8063019156455994, 'Number_of_Tickets': 1, 'Ticket_Similarity_Mean': 0.8126038312911987, 'Solution_Coherence': 1.0, 'Representative_Ticket': 'INC67571325', 'Representative_Solution': 'Hello Oussama,\n\nPlease note the new password : Focus@147258369@', 'Evidence_Tickets': [{'Number': 'INC67571325', 'Short Description': 'MA - EXP - RESET PASSWORD LDAP : zelhadda', 'Ticket_Similarity': 0.8126038312911987}]}


In [39]:
# ==========================================
# GÉNÉRATION DE RECOMMANDATION
# VERSION SÉCURISÉE
# ==========================================

import re


def sanitize_solution_text(text):
    """
    Supprime les informations potentiellement sensibles
    avant de construire la recommandation.
    """

    if text is None:
        return ""

    text = str(text)

    # --------------------------------------
    # Masquage des mots de passe
    # --------------------------------------

    text = re.sub(
        r'(?i)(password|mot de passe)\s*[:=]\s*.+',
        r'\1: [INFORMATION SENSIBLE MASQUÉE]',
        text
    )

    # --------------------------------------
    # Masquage des credentials explicites
    # --------------------------------------

    text = re.sub(
        r'(?i)(login|username|user)\s*[:=]\s*\S+',
        r'\1: [INFORMATION SENSIBLE MASQUÉE]',
        text
    )

    # --------------------------------------
    # Nettoyage
    # --------------------------------------

    text = re.sub(
        r'\n+',
        ' ',
        text
    )

    return text.strip()


def generate_recommendation_from_evidence(evidence):
    """
    Génère une recommandation en langage naturel
    à partir des évidences historiques.

    La solution historique n'est PAS copiée directement.
    """

    if evidence is None:
        return None

    if not isinstance(evidence, dict):
        return None

    # --------------------------------------
    # Récupération des évidences
    # --------------------------------------

    evidence_tickets = evidence.get(
        "Evidence_Tickets",
        []
    )

    if not evidence_tickets:
        return (
            "Aucune solution historique suffisamment "
            "pertinente n'a été identifiée."
        )

    # --------------------------------------
    # Récupérer les solutions correspondantes
    # --------------------------------------

    evidence_numbers = [
        item["Number"]
        for item in evidence_tickets
        if "Number" in item
    ]

    evidence_data = knowledge_base[
        knowledge_base["Number"].isin(
            evidence_numbers
        )
    ].copy()

    if evidence_data.empty:
        return (
            "Aucune solution historique exploitable "
            "n'a été trouvée."
        )

    # --------------------------------------
    # Nettoyage des solutions
    # --------------------------------------

    evidence_data["Clean_Solution"] = (
        evidence_data["Solution"]
        .fillna("")
        .apply(sanitize_solution_text)
    )

    # --------------------------------------
    # Construire une recommandation naturelle
    # --------------------------------------

    recommendations = []

    for solution in evidence_data[
        "Clean_Solution"
    ]:

        if not solution:
            continue

        solution_lower = solution.lower()

        # ------------------------------
        # Actions techniques courantes
        # ------------------------------

        if (
            "driver" in solution_lower
            and (
                "update" in solution_lower
                or "updated" in solution_lower
            )
        ):
            recommendations.append(
                "Vérifier puis mettre à jour le pilote "
                "de l'équipement concerné."
            )

        elif (
            "printer" in solution_lower
            and (
                "setup" in solution_lower
                or "configured" in solution_lower
                or "configuration" in solution_lower
            )
        ):
            recommendations.append(
                "Vérifier la configuration de l'imprimante "
                "et appliquer la configuration appropriée."
            )

        elif (
            "recreated" in solution_lower
            or "created" in solution_lower
        ):
            recommendations.append(
                "Vérifier la configuration de la ressource "
                "et la recréer si nécessaire."
            )

        elif (
            "installed" in solution_lower
            or "installation" in solution_lower
        ):
            recommendations.append(
                "Vérifier l'installation du composant concerné "
                "et procéder à sa réinstallation si nécessaire."
            )

        elif (
            "restart" in solution_lower
            or "restarted" in solution_lower
        ):
            recommendations.append(
                "Redémarrer le composant ou service concerné "
                "puis vérifier son fonctionnement."
            )

        elif (
            "reset" in solution_lower
            and "password" in solution_lower
        ):
            recommendations.append(
                "Effectuer une réinitialisation du mot de passe "
                "de l'utilisateur concerné via la procédure "
                "sécurisée appropriée."
            )

        elif (
            "resolved" in solution_lower
            or "fixed" in solution_lower
            or "working now" in solution_lower
        ):
            recommendations.append(
                "Vérifier la configuration du composant concerné "
                "et appliquer les corrections appropriées."
            )

    # --------------------------------------
    # Supprimer les doublons
    # --------------------------------------

    recommendations = list(
        dict.fromkeys(recommendations)
    )

    # --------------------------------------
    # Si une action identifiable existe
    # --------------------------------------

    if recommendations:

        return (
            "Sur la base des incidents historiques similaires, "
            "il est recommandé de :\n\n"
            + "\n".join(
                f"- {recommendation}"
                for recommendation in recommendations[:3]
            )
        )

    # --------------------------------------
    # Fallback
    # --------------------------------------

    return (
        "Des incidents historiques similaires ont été "
        "identifiés. Il est recommandé d'examiner leur "
        "résolution et de vérifier la configuration du "
        "composant concerné avant d'appliquer une correction."
    )

In [40]:
recommendation_test = (
    generate_recommendation_from_evidence(
        evidence
    )
)

print("=" * 60)
print("NOUVELLE RECOMMANDATION")
print("=" * 60)

print(recommendation_test)

NOUVELLE RECOMMANDATION
Sur la base des incidents historiques similaires, il est recommandé de :

- Vérifier la configuration de l'imprimante et appliquer la configuration appropriée.
- Vérifier la configuration du composant concerné et appliquer les corrections appropriées.
- Vérifier puis mettre à jour le pilote de l'équipement concerné.


In [41]:
recommendation_ldap = (
    generate_recommendation_from_evidence(
        evidence
    )
)

print("=" * 60)
print("TEST LDAP")
print("=" * 60)

print(recommendation_ldap)

TEST LDAP
Sur la base des incidents historiques similaires, il est recommandé de :

- Vérifier la configuration de l'imprimante et appliquer la configuration appropriée.
- Vérifier la configuration du composant concerné et appliquer les corrections appropriées.
- Vérifier puis mettre à jour le pilote de l'équipement concerné.


In [42]:
# ==========================================
# TEST LDAP
# ========
#==================================

ldap_query = """
MA - EXP - RESET PASSWORD LDAP : zelhadda
New account / modification / removal:
Modification of password
"""

ldap_result = recommend_solution(
    ldap_query,
    top_k=20,
    cluster_threshold=0.75
)

print("=" * 60)
print("TEST LDAP")
print("=" * 60)

print("STATUS :", ldap_result["status"])

print("\nEVIDENCE :")
print(ldap_result["evidence"])

print("\nRECOMMANDATION :")
print(ldap_result["recommendation"])

TEST LDAP
STATUS : SUCCESS

EVIDENCE :
{'Query': '\nMA - EXP - RESET PASSWORD LDAP : zelhadda\nNew account / modification / removal:\nModification of password\n', 'Winning_Cluster': 3, 'Cluster_Score': 0.9112799167633057, 'Number_of_Tickets': 1, 'Ticket_Similarity_Mean': 0.8225598335266113, 'Solution_Coherence': 1.0, 'Representative_Ticket': 'INC67571325', 'Representative_Solution': 'Hello Oussama,\n\nPlease note the new password : Focus@147258369@', 'Evidence_Tickets': [{'Number': 'INC67571325', 'Short Description': 'MA - EXP - RESET PASSWORD LDAP : zelhadda', 'Ticket_Similarity': 0.8225598335266113}]}

RECOMMANDATION :
Hello Oussama,

Please note the new password : Focus@147258369@


In [46]:
# ==========================================
# 7. GÉNÉRATION DE RECOMMANDATION NATURELLE
# ==========================================

def generate_natural_recommendation(evidence):
    """
    Génère une recommandation générique en langage naturel
    à partir des évidences produites par le pipeline.

    Les solutions historiques sont utilisées comme preuves,
    mais les informations sensibles ou spécifiques ne sont
    jamais recopiées dans la recommandation.
    """

    if evidence is None or not isinstance(evidence, dict):
        return (
            "Aucune recommandation fiable ne peut être générée."
        )

    # --------------------------------------
    # Récupérer la solution représentative
    # --------------------------------------

    representative_solution = str(
        evidence.get(
            "Representative_Solution",
            ""
        )
    ).strip()

    # --------------------------------------
    # Récupérer les descriptions des tickets
    # --------------------------------------

    evidence_tickets = evidence.get(
        "Evidence_Tickets",
        []
    )

    descriptions = []

    for ticket in evidence_tickets:

        description = ticket.get(
            "Short Description"
        )

        if description:
            descriptions.append(
                str(description)
            )

    # Texte utilisé pour identifier le contexte
    context_text = (
        representative_solution
        + " "
        + " ".join(descriptions)
    ).lower()

    # --------------------------------------
    # Génération de la recommandation
    # --------------------------------------

    recommendation_steps = []

    # ======================================
    # CAS IMPRIMANTE
    # ======================================

    if any(
        word in context_text
        for word in [
            "printer",
            "printing",
            "print",
            "driver"
        ]
    ):

        recommendation_steps = [

            "Vérifier la configuration de l'imprimante concernée.",

            "Vérifier puis mettre à jour ou réinstaller le pilote de l'imprimante si nécessaire.",

            "Effectuer un test d'impression afin de confirmer la résolution du problème."
        ]

    # ======================================
    # CAS LDAP / MOT DE PASSE
    # ======================================

    elif any(
        word in context_text
        for word in [
            "password",
            "ldap",
            "reset password"
        ]
    ):

        recommendation_steps = [

            "Procéder à la réinitialisation ou à la modification du mot de passe du compte concerné.",

            "Vérifier ensuite que l'utilisateur peut correctement s'authentifier avec ses nouveaux identifiants."
        ]

    # ======================================
    # CAS COMPTE / ACCÈS
    # ======================================

    elif any(
        word in context_text
        for word in [
            "account",
            "access",
            "login"
        ]
    ):

        recommendation_steps = [

            "Vérifier les droits et la configuration du compte concerné.",

            "Appliquer les corrections nécessaires sur les paramètres d'accès.",

            "Effectuer un test de connexion afin de confirmer le rétablissement de l'accès."
        ]

    # ======================================
    # CAS CONFIGURATION
    # ======================================

    elif any(
        word in context_text
        for word in [
            "configuration",
            "configured",
            "setup",
            "settings"
        ]
    ):

        recommendation_steps = [

            "Vérifier la configuration du composant concerné.",

            "Appliquer la configuration appropriée puis vérifier le fonctionnement du service."
        ]

    # ======================================
    # CAS INSTALLATION
    # ======================================

    elif any(
        word in context_text
        for word in [
            "installed",
            "installation",
            "reinstalled"
        ]
    ):

        recommendation_steps = [

            "Vérifier l'installation du composant concerné.",

            "Réinstaller ou compléter sa configuration si nécessaire.",

            "Vérifier ensuite que le service fonctionne correctement."
        ]

    # ======================================
    # CAS MISE À JOUR
    # ======================================

    elif any(
        word in context_text
        for word in [
            "updated",
            "update",
            "upgraded"
        ]
    ):

        recommendation_steps = [

            "Vérifier la version actuelle du composant concerné.",

            "Appliquer la mise à jour appropriée puis vérifier son bon fonctionnement."
        ]

    # ======================================
    # CAS GÉNÉRIQUE
    # ======================================

    else:

        recommendation_steps = [

            "Analyser la configuration du composant concerné afin d'identifier la cause du problème.",

            "Appliquer la correction appropriée en fonction du diagnostic.",

            "Effectuer un test afin de confirmer la résolution de l'incident."
        ]

    # --------------------------------------
    # Construction finale
    # --------------------------------------

    recommendation = (
        "Sur la base des incidents historiques similaires, "
        "il est recommandé de :\n\n"
    )

    for step in recommendation_steps:

        recommendation += f"- {step}\n"

    return recommendation.strip()

In [52]:
import inspect

print(inspect.signature(build_recommendation_evidence))

(query, best_result, ranked_clusters, clusters, actionable, top_n_tickets=5)


In [57]:
# ==========================================
# PIPELINE COMPLET DE RECOMMANDATION
# ==========================================

def recommend_incident(
    query,
    top_k=20,
    cluster_threshold=0.75
):
    """
    Pipeline complet de recommandation.

    Étapes :
    1. Recherche des tickets similaires
    2. Filtrage des solutions ACTIONABLE
    3. Embedding des solutions
    4. Similarité entre solutions
    5. Clustering
    6. Scoring des clusters
    7. Classement
    8. Sélection du cluster gagnant
    9. Construction des évidences
    10. Génération d'une recommandation naturelle
    """

    # ======================================
    # 1. Recherche des tickets similaires
    # ======================================

    candidates = search_similar_tickets(
        query,
        top_k=top_k
    )

    # ======================================
    # 2. Filtrage ACTIONABLE
    # ======================================

    actionable = filter_actionable_candidates(
        candidates
    )

    if actionable.empty:

        return {
            "status": "NO_ACTIONABLE_SOLUTION",
            "query": query,
            "recommendation": None,
            "evidence": None
        }

    # ======================================
    # 3. Embeddings des solutions
    # ======================================

    solution_texts = (
        actionable["Solution"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    candidate_solution_embeddings = model.encode(
        solution_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # ======================================
    # 4. Similarité entre solutions
    # ======================================

    solution_matrix = cosine_similarity(
        candidate_solution_embeddings
    )

    solution_similarity_matrix = pd.DataFrame(
        solution_matrix,
        index=actionable["Number"],
        columns=actionable["Number"]
    )

    # ======================================
    # 5. Clustering
    # ======================================

    clusters = cluster_similar_solutions(
        solution_similarity_matrix,
        threshold=cluster_threshold
    )

    # ======================================
    # 6. Scoring des clusters
    # ======================================

    cluster_scores = score_solution_clusters(
        actionable,
        clusters,
        solution_similarity_matrix
    )

    if cluster_scores.empty:

        return {
            "status": "NO_VALID_CLUSTER",
            "query": query,
            "recommendation": None,
            "evidence": None
        }

    # ======================================
    # 7. Classement des clusters
    # ======================================

    ranked_clusters = rank_solution_clusters(
        cluster_scores
    )

    # ======================================
    # 8. Cluster gagnant
    # ======================================

    best_result = (
        ranked_clusters.iloc[0].to_dict()
    )

    # ======================================
    # 9. Construction des évidences
    # ======================================

    evidence = build_recommendation_evidence(
        query,
        best_result,
        ranked_clusters,
        clusters,
        actionable,
        top_n_tickets=5
    )

    # ======================================
    # 10. Recommandation naturelle
    # ======================================

    recommendation = generate_natural_recommendation(
        query,
        best_result,
        ranked_clusters,
        clusters,
        actionable,
        top_n_tickets=5
    )

    # ======================================
    # 11. Informations du cluster gagnant
    # ======================================

    best_cluster = int(
        best_result["Cluster"]
    )

    # ======================================
    # 12. Résultat final
    # ======================================

    return {

        "status":
            "SUCCESS",

        "query":
            query,

        "recommendation":
            recommendation,

        "winning_cluster":
            best_cluster,

        "cluster_score":
            best_result[
                "Cluster_Final_Score"
            ],

        "evidence":
            evidence,

        "candidates":
            actionable,

        "ranked_clusters":
            ranked_clusters
    }

In [60]:
# ==========================================
# PIPELINE COMPLET DE RECOMMANDATION
# ==========================================

def recommend_incident(
    query,
    top_k=20,
    cluster_threshold=0.75
):
    """
    Pipeline complet de recommandation.

    Étapes :
    1. Recherche des tickets similaires
    2. Filtrage des solutions ACTIONABLE
    3. Embedding des solutions
    4. Similarité entre solutions
    5. Clustering
    6. Scoring des clusters
    7. Classement
    8. Sélection du cluster gagnant
    9. Sélection du ticket représentatif
    10. Construction des évidences
    11. Génération d'une recommandation naturelle
    """

    # ======================================
    # 1. Recherche des tickets similaires
    # ======================================

    candidates = search_similar_tickets(
        query,
        top_k=top_k
    )

    # ======================================
    # 2. Filtrage ACTIONABLE
    # ======================================

    actionable = filter_actionable_candidates(
        candidates
    )

    if actionable.empty:

        return {
            "status": "NO_ACTIONABLE_SOLUTION",
            "query": query,
            "recommendation": None,
            "evidence": None
        }

    # ======================================
    # 3. Embeddings des solutions
    # ======================================

    solution_texts = (
        actionable["Solution"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    candidate_solution_embeddings = model.encode(
        solution_texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # ======================================
    # 4. Similarité entre solutions
    # ======================================

    solution_matrix = cosine_similarity(
        candidate_solution_embeddings
    )

    solution_similarity_matrix = pd.DataFrame(
        solution_matrix,
        index=actionable["Number"],
        columns=actionable["Number"]
    )

    # ======================================
    # 5. Clustering
    # ======================================

    clusters = cluster_similar_solutions(
        solution_similarity_matrix,
        threshold=cluster_threshold
    )

    # ======================================
    # 6. Scoring des clusters
    # ======================================

    cluster_scores = score_solution_clusters(
        actionable,
        clusters,
        solution_similarity_matrix
    )

    if cluster_scores.empty:

        return {
            "status": "NO_VALID_CLUSTER",
            "query": query,
            "recommendation": None,
            "evidence": None
        }

    # ======================================
    # 7. Classement des clusters
    # ======================================

    ranked_clusters = rank_solution_clusters(
        cluster_scores
    )

    # ======================================
    # 8. Cluster gagnant
    # ======================================

    best_cluster = int(
        ranked_clusters.iloc[0]["Cluster"]
    )

    # ======================================
    # 9. Tickets du cluster gagnant
    # ======================================

    winning_cluster_numbers = clusters.loc[
        clusters["Cluster"] == best_cluster,
        "Number"
    ].tolist()

    winning_cluster_data = actionable[
        actionable["Number"].isin(
            winning_cluster_numbers
        )
    ].copy()

    # ======================================
    # 10. Sélection du ticket représentatif
    # ======================================

    if len(winning_cluster_data) == 1:

        representative = (
            winning_cluster_data.iloc[0]
        )

        representative_ticket = (
            representative["Number"]
        )

        representative_solution = (
            representative["Solution"]
        )

        internal_similarity = 1.0

    else:

        matrix = solution_similarity_matrix.loc[
            winning_cluster_numbers,
            winning_cluster_numbers
        ]

        mean_similarities = {}

        for ticket_number in winning_cluster_numbers:

            others = [
                t
                for t in winning_cluster_numbers
                if t != ticket_number
            ]

            mean_similarities[
                ticket_number
            ] = matrix.loc[
                ticket_number,
                others
            ].mean()

        representative_ticket = max(
            mean_similarities,
            key=mean_similarities.get
        )

        representative = winning_cluster_data[
            winning_cluster_data["Number"]
            == representative_ticket
        ].iloc[0]

        representative_solution = (
            representative["Solution"]
        )

        internal_similarity = (
            mean_similarities[
                representative_ticket
            ]
        )

    # ======================================
    # 11. Construire best_result
    # ======================================
    #
    # IMPORTANT :
    # build_recommendation_evidence()
    # attend explicitement ces informations.
    #

    best_cluster_row = (
        ranked_clusters[
            ranked_clusters["Cluster"]
            == best_cluster
        ].iloc[0]
    )

    best_result = {

        "Cluster":
            best_cluster,

        "Cluster_Final_Score":
            best_cluster_row[
                "Cluster_Final_Score"
            ],

        "Representative_Ticket":
            representative_ticket,

        "Representative_Solution":
            representative_solution,

        "Ticket_Similarity":
            representative[
                "Ticket_Similarity"
            ],

        "Internal_Similarity":
            internal_similarity,

        "Number_of_Tickets":
            best_cluster_row[
                "Number_of_Tickets"
            ],

        "Ticket_Similarity_Mean":
            best_cluster_row[
                "Ticket_Similarity_Mean"
            ],

        "Solution_Coherence":
            best_cluster_row[
                "Solution_Coherence"
            ]
    }

    # ======================================
    # 12. Construction des évidences
    # ======================================

    evidence = build_recommendation_evidence(
        query,
        best_result,
        ranked_clusters,
        clusters,
        actionable
    )

    # ======================================
    # 13. Recommandation naturelle
    # ======================================

    recommendation = (
        generate_natural_recommendation(
            evidence
        )
    )

    # ======================================
    # 14. Résultat final
    # ======================================

    return {

        "status":
            "SUCCESS",

        "query":
            query,

        "recommendation":
            recommendation,

        "winning_cluster":
            best_cluster,

        "cluster_score":
            best_cluster_row[
                "Cluster_Final_Score"
            ],

        "representative_ticket":
            representative_ticket,

        "representative_solution":
            representative_solution,

        "evidence":
            evidence,

        "candidates":
            actionable,

        "ranked_clusters":
            ranked_clusters
    }

In [62]:
# ==========================================
# TEST DU PIPELINE COMPLET - GCA
# ==========================================

gca_query = """
GCA printer is not working and delivery sheets cannot be printed.
"""

gca_result = recommend_incident(
    gca_query
)

print("=" * 70)
print("PIPELINE COMPLET - TEST GCA")
print("=" * 70)

print("\nSTATUS :")
print(gca_result["status"])

print("\nRECOMMANDATION :")
print(gca_result["recommendation"])

print("\nTICKET REPRESENTATIF :")
print(gca_result.get("representative_ticket", "Voir evidence"))

print("\nSOLUTION HISTORIQUE :")
print(gca_result.get("representative_solution", "Voir evidence"))

print("\nSCORE DU CLUSTER :")
print(gca_result["cluster_score"])

print("\nÉVIDENCES :")
print(gca_result["evidence"])

PIPELINE COMPLET - TEST GCA

STATUS :
SUCCESS

RECOMMANDATION :
Sur la base des incidents historiques similaires, il est recommandé de :

- Vérifier la configuration de l'imprimante concernée.
- Vérifier puis mettre à jour ou réinstaller le pilote de l'imprimante si nécessaire.
- Effectuer un test d'impression afin de confirmer la résolution du problème.

TICKET REPRESENTATIF :
INC65363499

SOLUTION HISTORIQUE :
Printer setup has been updated and it is working now.

Resolved.


SCORE DU CLUSTER :
0.7931201040744782

ÉVIDENCES :
{'Query': '\nGCA printer is not working and delivery sheets cannot be printed.\n', 'Winning_Cluster': 2, 'Cluster_Score': 0.7931201040744782, 'Number_of_Tickets': 3, 'Ticket_Similarity_Mean': 0.6902837753295898, 'Solution_Coherence': 0.8265940546989441, 'Representative_Ticket': 'INC65363499', 'Representative_Solution': 'Printer setup has been updated and it is working now.\n\nResolved.\n', 'Evidence_Tickets': [{'Number': 'INC66310792', 'Short Description': 'Issu

In [63]:
# ==========================================
# TEST PIPELINE COMPLET - LDAP
# ==========================================

ldap_query = """
MA - EXP - RESET PASSWORD LDAP : zelhadda
New account / modification / removal:
Modification of password
"""

ldap_result = recommend_incident(
    ldap_query
)

print("=" * 70)
print("PIPELINE COMPLET - TEST LDAP")
print("=" * 70)

print("\nSTATUS :")
print(ldap_result["status"])

print("\nRECOMMANDATION :")
print(ldap_result["recommendation"])

print("\nTICKET REPRESENTATIF :")
print(ldap_result["representative_ticket"])

print("\nSOLUTION HISTORIQUE :")
print(ldap_result["representative_solution"])

print("\nSCORE DU CLUSTER :")
print(ldap_result["cluster_score"])

print("\nÉVIDENCES :")
print(ldap_result["evidence"])

PIPELINE COMPLET - TEST LDAP

STATUS :
SUCCESS

RECOMMANDATION :
Sur la base des incidents historiques similaires, il est recommandé de :

- Procéder à la réinitialisation ou à la modification du mot de passe du compte concerné.
- Vérifier ensuite que l'utilisateur peut correctement s'authentifier avec ses nouveaux identifiants.

TICKET REPRESENTATIF :
INC67571325

SOLUTION HISTORIQUE :
Hello Oussama,

Please note the new password : Focus@147258369@

SCORE DU CLUSTER :
0.9112799167633057

ÉVIDENCES :
{'Query': '\nMA - EXP - RESET PASSWORD LDAP : zelhadda\nNew account / modification / removal:\nModification of password\n', 'Winning_Cluster': 3, 'Cluster_Score': 0.9112799167633057, 'Number_of_Tickets': 1, 'Ticket_Similarity_Mean': 0.8225598335266113, 'Solution_Coherence': 1.0, 'Representative_Ticket': 'INC67571325', 'Representative_Solution': 'Hello Oussama,\n\nPlease note the new password : Focus@147258369@', 'Evidence_Tickets': [{'Number': 'INC67571325', 'Short Description': 'MA - EXP 

In [64]:
# ==========================================
# EXPLORATION DES CAS POUR LE 3ÈME TEST
# ==========================================

display(
    knowledge_base[
        [
            "Number",
            "Short Description",
            "Solution",
            "solution_type"
        ]
    ].sample(10, random_state=42)
)

,Number,Short Description,Solution,solution_type
727,INC71877391,MA - EXP - After several attempts to access PO...,RESOLVED,GENERIC
907,INC69304265,MA - EXP - esecure f.jouki@enosis.ma pour le ...,Done.,GENERIC
1547,INC66765753,MA - EXP - Printer issue,Resolved,GENERIC
240,INC72044971,MA - EXP - Correction of Phone Number on Invoi...,Fixed by Chetan,ACTIONABLE
686,INC67571325,MA - EXP - RESET PASSWORD LDAP : zelhadda,"Hello Oussama,\n\nPlease note the new password...",ACTIONABLE
1541,INC67120356,MA - EXP - IMPRIMANTE XEROX CIA TONER EMPTY,solved,GENERIC
480,INC71245757,MA - EXP - MYDHL+ URGENT,resolved,GENERIC
668,INC69678251,MA - EXP - Acces to mydhl+ and receive code es...,resolved,GENERIC
1652,INC71940783,MA - EXP - reactivation of GCA,resolved CASCAS,ACTIONABLE
1025,INC68607375,MA - EXP - Probleme des nouveaux Carte SIM orange,Issue under Provider investigations,ACTIONABLE


In [65]:
# ==========================================
# 3ÈME CAS DE TEST
# ==========================================

test_number = "INC72044971"

test_case = knowledge_base[
    knowledge_base["Number"] == test_number
].iloc[0]

print("=" * 70)
print("TICKET DE TEST")
print("=" * 70)

print("\nNumber :")
print(test_case["Number"])

print("\nShort Description :")
print(test_case["Short Description"])

print("\nDescription :")
print(test_case["Description"])

print("\nSolution réelle :")
print(test_case["Solution"])

print("\nSolution type :")
print(test_case["solution_type"])

TICKET DE TEST

Number :
INC72044971

Short Description :
MA - EXP - Correction of Phone Number on Invoices and Dunning Letters

Description :
Dear Nabil, Chetan

Hope you are doing well,

Please be informed that the Phone number mentioned in our invoices and dunning is incorrect , below the correct direct number of QH


Contact preference: phone
Phone: 212 20000 0522321531

Solution réelle :
Fixed by Chetan

Solution type :
ACTIONABLE


In [66]:
# ==========================================
# TEST 3 - CORRECTION NUMÉRO DE TÉLÉPHONE
# ==========================================

test3_query = """
The phone number displayed on invoices and dunning letters is incorrect.
The correct phone number needs to be updated.
"""

In [67]:
test3_result = recommend_incident(
    test3_query
)

print("=" * 70)
print("PIPELINE COMPLET - TEST 3")
print("=" * 70)

print("\nSTATUS :")
print(test3_result["status"])

print("\nRECOMMANDATION :")
print(test3_result["recommendation"])

print("\nTICKET REPRESENTATIF :")
print(test3_result["representative_ticket"])

print("\nSOLUTION HISTORIQUE :")
print(test3_result["representative_solution"])

print("\nSCORE DU CLUSTER :")
print(test3_result["cluster_score"])

print("\nÉVIDENCES :")
print(test3_result["evidence"])

PIPELINE COMPLET - TEST 3

STATUS :
SUCCESS

RECOMMANDATION :
Sur la base des incidents historiques similaires, il est recommandé de :

- Analyser la configuration du composant concerné afin d'identifier la cause du problème.
- Appliquer la correction appropriée en fonction du diagnostic.
- Effectuer un test afin de confirmer la résolution de l'incident.

TICKET REPRESENTATIF :
INC72044971

SOLUTION HISTORIQUE :
Fixed by Chetan

SCORE DU CLUSTER :
0.8694075345993042

ÉVIDENCES :
{'Query': '\nThe phone number displayed on invoices and dunning letters is incorrect.\nThe correct phone number needs to be updated.\n', 'Winning_Cluster': 3, 'Cluster_Score': 0.8694075345993042, 'Number_of_Tickets': 1, 'Ticket_Similarity_Mean': 0.7388150691986084, 'Solution_Coherence': 1.0, 'Representative_Ticket': 'INC72044971', 'Representative_Solution': 'Fixed by Chetan', 'Evidence_Tickets': [{'Number': 'INC72044971', 'Short Description': 'MA - EXP - Correction of Phone Number on Invoices and Dunning Letter

In [68]:
# ==========================================
# TEST 4 — RECHERCHE D'UNE SOLUTION RICHE
# ==========================================

test_rich_solutions = knowledge_base[
    (knowledge_base["solution_type"] == "ACTIONABLE")
    &
    (knowledge_base["Solution"].notna())
].copy()

test_rich_solutions["solution_length"] = (
    test_rich_solutions["Solution"]
    .astype(str)
    .str.len()
)

display(
    test_rich_solutions[
        [
            "Number",
            "Short Description",
            "Solution",
            "solution_type",
            "solution_length"
        ]
    ]
    .sort_values("solution_length", ascending=False)
    .head(10)
)

,Number,Short Description,Solution,solution_type,solution_length
1979,INC61302608,Add printers on THOR app,done all mentioned printers installed on serve...,ACTIONABLE,5871
1827,INC65410555,Request to change printer model for GCA printe...,As per the request printer driver updated \n\n...,ACTIONABLE,5583
2194,INC62486177,MA - EXP - MS Outlook || Unable to receive ext...,Dear User:\nCheck the domails on the domains t...,ACTIONABLE,5182
1964,INC61363315,Change IP ADR,The printers ip address has been changed as re...,ACTIONABLE,4829
2017,INC61209735,Change THOR Printer IP address,The printer has been created as requested.\n\n...,ACTIONABLE,4431
1942,INC61241300,Change GCA7 Printer IP address,Mentioned printers ip change completed in all ...,ACTIONABLE,2103
2211,INC66235974,MA-EXP- Issue on data transmission from MAD or...,as confirmed by user. shipments is confirmed l...,ACTIONABLE,2061
2028,INC61209815,Change Printer IP adr on THOR system,updated\nMYKULLSPC000578\ndevice for MATNGKEN0...,ACTIONABLE,1791
1947,INC61687260,Change IP address on THOR,IP 23.82.236.234 was already set on all server...,ACTIONABLE,1655
1884,INC67124646,"Slowness in MA SDWAN sites : EMTO-MA-EXP-001, ...",BT Time of Resolution : 2025-12-03T11:05:30Z\n...,ACTIONABLE,1423


In [69]:
# ==========================================
# TEST 4 — TICKET AVEC SOLUTION DÉTAILLÉE
# ==========================================

test_ticket = knowledge_base[
    knowledge_base["Number"] == "INC61302608"
].iloc[0]

print("=" * 70)
print("TICKET DE TEST 4")
print("=" * 70)

print("\nNumber :")
print(test_ticket["Number"])

print("\nShort Description :")
print(test_ticket["Short Description"])

print("\nDescription :")
print(test_ticket["Description"])

print("\nSolution réelle :")
print(test_ticket["Solution"])

print("\nSolution type :")
print(test_ticket["solution_type"])

print("\nLongueur solution :")
print(len(str(test_ticket["Solution"])))

TICKET DE TEST 4

Number :
INC61302608

Short Description :
Add printers on THOR app

Description :
Dears,

Could you please add the following printers ::

MATNGKEN03 : 23.82.61.234 
MATNGKEN04 : 23.82.61.235 

MARBARAK03 : 23.82.50.234 
MARBARAK04 : 23.82.50.235 
 
MARBAAGA03 : 23.82.240.234 
MARBAAGA04 : 23.82.240.235 
 
MARBAGLZ02 : 23.82.244.234


Thanks in ad

Solution réelle :
done all mentioned printers installed on servers.

MYKULLSPC000578
device for MATNGKEN03: beh:/1/0/60/socket://23.82.61.234:9100
MATNGKEN03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MATNGKEN03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MATNGKEN04: beh:/1/0/60/socket://23.82.61.235:9100
MATNGKEN04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MATNGKEN04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MARBARAK03: beh:/1/0/60/socket://23.82.50.234:9100
MARBARAK03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer

In [70]:
# ==========================================
# TEST 4 — PIPELINE SUR UN TICKET RÉEL
# ==========================================

test4_query = f"""
{test_ticket["Short Description"]}

{test_ticket["Description"]}
"""

test4_result = recommend_incident(
    test4_query
)

print("=" * 70)
print("PIPELINE COMPLET - TEST 4")
print("=" * 70)

print("\nSTATUS :")
print(test4_result["status"])

if test4_result["status"] == "SUCCESS":

    print("\nRECOMMANDATION :")
    print(test4_result["recommendation"])

    print("\nTICKET REPRESENTATIF :")
    print(test4_result["representative_ticket"])

    print("\nSOLUTION HISTORIQUE RETROUVÉE :")
    print(test4_result["representative_solution"])

    print("\nSCORE DU CLUSTER :")
    print(test4_result["cluster_score"])

    print("\nÉVIDENCES :")
    for evidence_ticket in test4_result["evidence"]["Evidence_Tickets"]:
        print(
            evidence_ticket["Number"],
            " | ",
            evidence_ticket["Ticket_Similarity"]
        )

else:

    print("\nAucune recommandation disponible.")
    print(test4_result)

PIPELINE COMPLET - TEST 4

STATUS :
SUCCESS

RECOMMANDATION :
Sur la base des incidents historiques similaires, il est recommandé de :

- Vérifier la configuration de l'imprimante concernée.
- Vérifier puis mettre à jour ou réinstaller le pilote de l'imprimante si nécessaire.
- Effectuer un test d'impression afin de confirmer la résolution du problème.

TICKET REPRESENTATIF :
INC61302569

SOLUTION HISTORIQUE RETROUVÉE :
Completed, printers installed on demanded servers:
MAKENPR4
----------
device for MAKENPR4: beh:/1/0/60/socket://23.82.61.234:9100
MAKENPR4 accepting requests since Wed 16 Oct 2024 10:33:21 PM GMT
printer MAKENPR4 is idle.  enabled since Wed 16 Oct 2024 10:33:21 PM GMT
MAKENPR3
----------
device for MAKENPR3: beh:/1/0/60/socket://23.82.61.235:9100
MAKENPR3 accepting requests since Wed 16 Oct 2024 10:33:21 PM GMT
printer MAKENPR3 is idle.  enabled since Wed 16 Oct 2024 10:33:21 PM GMT
MARAKPR4
----------
device for MARAKPR4: beh:/1/0/60/socket://23.82.50.234:9100
MARAKPR

In [71]:
# ==========================================
# DIAGNOSTIC DU CLUSTER - TEST 4
# ==========================================

print("=" * 70)
print("DIAGNOSTIC DU CLUSTER - INC61302608")
print("=" * 70)

# ------------------------------------------
# 1. Cluster auquel appartient INC61302608
# ------------------------------------------

ticket_number = "INC61302608"

ticket_cluster = clusters[
    clusters["Number"] == ticket_number
]

print("\nCLUSTER DU TICKET :")
print(ticket_cluster)

# ------------------------------------------
# 2. Score de ce cluster
# ------------------------------------------

if not ticket_cluster.empty:

    cluster_id = ticket_cluster.iloc[0]["Cluster"]

    print("\nID DU CLUSTER :", cluster_id)

    cluster_info = ranked_clusters[
        ranked_clusters["Cluster"] == cluster_id
    ]

    print("\nSCORE DU CLUSTER :")
    display(cluster_info)

    # --------------------------------------
    # 3. Tous les tickets du cluster
    # --------------------------------------

    cluster_numbers = clusters.loc[
        clusters["Cluster"] == cluster_id,
        "Number"
    ].tolist()

    cluster_tickets = actionable[
        actionable["Number"].isin(cluster_numbers)
    ].copy()

    cluster_tickets = cluster_tickets.sort_values(
        "Ticket_Similarity",
        ascending=False
    )

    print("\nTICKETS DU CLUSTER :")
    display(
        cluster_tickets[
            [
                "Number",
                "Short Description",
                "Ticket_Similarity",
                "Solution"
            ]
        ]
    )

else:

    print(
        "\nINC61302608 n'a pas été trouvé dans les clusters."
    )

DIAGNOSTIC DU CLUSTER - INC61302608

CLUSTER DU TICKET :
Empty DataFrame
Columns: [Number, Cluster]
Index: []

INC61302608 n'a pas été trouvé dans les clusters.


In [72]:
# ==========================================
# DIAGNOSTIC DES IDENTIFIANTS
# ==========================================

ticket_number = "INC61302608"

print("=" * 70)
print("DIAGNOSTIC DES IDENTIFIANTS")
print("=" * 70)

print("\n1. Dans actionable ?")

print(
    actionable[
        actionable["Number"].astype(str) == ticket_number
    ][["Number", "Short Description", "Ticket_Similarity"]]
)

print("\n2. Type de actionable['Number'] :")
print(actionable["Number"].dtype)

print("\n3. Type de clusters['Number'] :")
print(clusters["Number"].dtype)

print("\n4. INC61302608 dans clusters ?")

print(
    clusters[
        clusters["Number"].astype(str) == ticket_number
    ]
)

print("\n5. Nombre de tickets dans actionable :")
print(len(actionable))

print("\n6. Nombre de tickets dans clusters :")
print(len(clusters))

print("\n7. Premiers identifiants de clusters :")
print(clusters["Number"].head(10).tolist())

DIAGNOSTIC DES IDENTIFIANTS

1. Dans actionable ?
Empty DataFrame
Columns: [Number, Short Description, Ticket_Similarity]
Index: []

2. Type de actionable['Number'] :
str

3. Type de clusters['Number'] :
str

4. INC61302608 dans clusters ?
Empty DataFrame
Columns: [Number, Cluster]
Index: []

5. Nombre de tickets dans actionable :
10

6. Nombre de tickets dans clusters :
10

7. Premiers identifiants de clusters :
['INC66310792', 'INC65363499', 'INC66441487', 'INC64341450', 'INC65033159', 'INC64304524', 'INC65410555', 'INC61302569', 'INC61933011', 'INC65461130']


In [73]:
# ==========================================
# DIAGNOSTIC RECHERCHE - INC61302608
# ==========================================

query = """
Add printers on THOR app
"""

# Recherche sur toute la knowledge base
all_candidates = search_similar_tickets(
    query,
    top_k=len(knowledge_base)
)

target = all_candidates[
    all_candidates["Number"] == "INC61302608"
]

print("=" * 70)
print("DIAGNOSTIC RECHERCHE - INC61302608")
print("=" * 70)

if target.empty:
    print("INC61302608 introuvable dans la recherche.")
else:
    print(target[
        [
            "Number",
            "Short Description",
            "Solution",
            "Ticket_Similarity",
            "solution_type"
        ]
    ])

    position = (
        all_candidates.reset_index()
        .index[
            all_candidates.reset_index()["Number"]
            == "INC61302608"
        ]
    )

    if len(position) > 0:
        print("\nPosition dans la Knowledge Base triée :", position[0] + 1)

DIAGNOSTIC RECHERCHE - INC61302608
        Number         Short Description  \
1  INC61302608  Add printers on THOR app   

                                            Solution  Ticket_Similarity  \
1  done all mentioned printers installed on serve...           0.771933   

  solution_type  
1    ACTIONABLE  

Position dans la Knowledge Base triée : 2


In [74]:
thor_query = """
Add printers on THOR app.
The requested printers need to be installed on the servers.
"""

thor_result = recommend_incident(
    thor_query
)

print("=" * 70)
print("PIPELINE COMPLET - TEST THOR")
print("=" * 70)

print("\nSTATUS :")
print(thor_result["status"])

print("\nRECOMMANDATION :")
print(thor_result["recommendation"])

print("\nTICKET REPRESENTATIF :")
print(thor_result["representative_ticket"])

print("\nSOLUTION HISTORIQUE :")
print(thor_result["representative_solution"])

print("\nSCORE DU CLUSTER :")
print(thor_result["cluster_score"])

print("\nÉVIDENCES :")
print(thor_result["evidence"])

PIPELINE COMPLET - TEST THOR

STATUS :
SUCCESS

RECOMMANDATION :
Sur la base des incidents historiques similaires, il est recommandé de :

- Vérifier la configuration de l'imprimante concernée.
- Vérifier puis mettre à jour ou réinstaller le pilote de l'imprimante si nécessaire.
- Effectuer un test d'impression afin de confirmer la résolution du problème.

TICKET REPRESENTATIF :
INC61302608

SOLUTION HISTORIQUE :
done all mentioned printers installed on servers.

MYKULLSPC000578
device for MATNGKEN03: beh:/1/0/60/socket://23.82.61.234:9100
MATNGKEN03 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MATNGKEN03 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MATNGKEN04: beh:/1/0/60/socket://23.82.61.235:9100
MATNGKEN04 accepting requests since Thu 17 Oct 2024 08:10:21 AM GMT
printer MATNGKEN04 is idle.  enabled since Thu 17 Oct 2024 08:10:21 AM GMT
device for MARBARAK03: beh:/1/0/60/socket://23.82.50.234:9100
MARBARAK03 accepting requests since Thu 17 O

In [78]:
# Vérification des exemples ACTIONABLE
actionable = knowledge_base[
    knowledge_base["solution_type"] == "ACTIONABLE"
][["Number", "Short Description", "Solution"]]

display(
    actionable.sample(
        min(50, len(actionable)),
        random_state=42
    )
)

,Number,Short Description,Solution
1398,INC68419302,MA - EXP - Need a new GOALS scanner to support...,Damaged scanner will be changed asap
2160,INC68460070,Changing extension number,"changes complete, please test and reach me on ..."
867,INC70527596,"MA - EXP - CAA, PMT, GSA et WebCapture Access",Access granted.
1945,INC64400511,Change printer IP address on THOR app,The printer ip has been updated as requested.\...
1863,INC62703001,Avaya phone with Extension 51855135 not receiv...,"Extension has been re-configured, and issue is..."
1817,INC62188298,Need DCVS workstations configuration,in UMEX is configured:\n\nWorkstation RBARBACO...
2041,INC60986930,Re-enable MA DCE-IAS Flow,"as per Olivier, there is no changes current pr..."
1742,INC71910715,MA - EXP - New GSM Number,New SIM card provided
175,INC65064993,MA - EXP - request access to mct ( mouad saif),Access granted
1415,INC69940863,ecase access,access granted


In [79]:
# Vérification des NON_ACTIONABLE
non_actionable = knowledge_base[
    knowledge_base["solution_type"] == "NON_ACTIONABLE"
][["Number", "Short Description", "Solution"]]

display(
    non_actionable.sample(
        min(50, len(non_actionable)),
        random_state=42
    )
)

,Number,Short Description,Solution
2199,INC65986979,Request to grant Sherloc broker access,Done set the access for the users
2173,INC66051641,MA EXP Outlook Issue,"Dear user,\n\nDoing a message trace we see all..."
1290,INC67684905,MA - EXP - Esecure,Le necessaire est bien fait\n
2175,INC69490389,NGWS OSD Installation Failure MANOUWDA1667268...,Devices installing as usual
2116,INC62449976,MA-EXP-Issue with the CAA application,GWS Maintenance for today
1776,INC70670066,qTest access,Granted
1352,INC68197928,MA - EXP - GCPT +,Merci de voir avec aymane
2131,INC69756758,MA - EXP - Grafana,"Dear colleague, because the required informati..."
1933,INC62202218,Request access to DIMS,no approval is provided yet before raising thi...
1162,INC64687646,MA - EXP - NMIV DOWN,Functionality issue solved


In [80]:
# Vérification des NON_ACTIONABLE
non_actionable = knowledge_base[
    knowledge_base["solution_type"] == "NON_ACTIONABLE"
][["Number", "Short Description", "Solution"]]

display(
    non_actionable.sample(
        min(50, len(non_actionable)),
        random_state=42
    )
)

,Number,Short Description,Solution
2199,INC65986979,Request to grant Sherloc broker access,Done set the access for the users
2173,INC66051641,MA EXP Outlook Issue,"Dear user,\n\nDoing a message trace we see all..."
1290,INC67684905,MA - EXP - Esecure,Le necessaire est bien fait\n
2175,INC69490389,NGWS OSD Installation Failure MANOUWDA1667268...,Devices installing as usual
2116,INC62449976,MA-EXP-Issue with the CAA application,GWS Maintenance for today
1776,INC70670066,qTest access,Granted
1352,INC68197928,MA - EXP - GCPT +,Merci de voir avec aymane
2131,INC69756758,MA - EXP - Grafana,"Dear colleague, because the required informati..."
1933,INC62202218,Request access to DIMS,no approval is provided yet before raising thi...
1162,INC64687646,MA - EXP - NMIV DOWN,Functionality issue solved


In [86]:
# ==========================================
# CLASSIFICATION FINALE DES SOLUTIONS
# ==========================================

import re
import pandas as pd


def classify_solution(solution):
    """
    Classe une solution historique.

    Classes :
    - ACTIONABLE
    - GENERIC
    - ADMINISTRATIVE
    - NON_ACTIONABLE
    """

    if pd.isna(solution):
        return "NON_ACTIONABLE"

    s = str(solution).strip().lower()

    if not s:
        return "NON_ACTIONABLE"

    # ======================================
    # 1. ADMINISTRATIVE
    # ======================================

    administrative_patterns = [

        "duplicate",
        "duplicated",

        "close the ticket",

        "ticket to avoid",

        "no response from user",

        "regional ticket opened",

        "new ticket has been raised",

        "a new ticket has been raised",

        "ticket has been raised",

        "ticket opened",

        "data provided",

        "information provided",

        "wrongly routed",

        "cancelled",

        "cancelled by"
    ]

    if any(
        pattern in s
        for pattern in administrative_patterns
    ):
        return "ADMINISTRATIVE"

    # ======================================
    # 2. NON ACTIONABLE
    # ======================================

    non_actionable_patterns = [

        "issue under provider investigations",

        "provider is working on it",

        "provider is working on the issue",

        "no approval is provided",

        "required information needed to resolve",

        "required information was not provided",

        "please ask performance to provide",

        "please reach out to",

        "please contact",

        "please request access from",

        "not our responsibility",

        "no issue from our side",

        "waiting for",

        "waiting on",

        "pending",

        "under investigation",

        "changes ongoing",

        "will be changed",

        "will be configured",

        "will be installed"
    ]

    if any(
        pattern in s
        for pattern in non_actionable_patterns
    ):
        return "NON_ACTIONABLE"

    # ======================================
    # 3. GENERIC
    # ======================================

    generic_solutions = {

        "resolved",
        "resolved.",
        "resolved,",

        "solved",
        "solved.",

        "done",
        "done.",

        "fixed",
        "fixed.",

        "closed",
        "closed.",

        "completed",
        "completed.",

        "granted",

        "access granted",
        "access granted.",

        "necessary done",
        "necessary done.",

        "necessaey done.",

        "necessary changes are made.",

        "the necessary changes are made.",

        "change done",
        "change done.",

        "fixed1"
    }

    if s in generic_solutions:
        return "GENERIC"

    if re.match(
        r"^(resolved|fixed)\s+inc[\w-]+",
        s
    ):
        return "GENERIC"

    # ======================================
    # 4. ACTIONABLE
    # ======================================

    actionable_patterns = [

        "installed",

        "installation done",

        "created",

        "was created",

        "has been created",

        "configured",

        "configuration done",

        "updated",

        "has been updated",

        "was updated",

        "changed",

        "has been changed",

        "was changed",

        "replaced",

        "has been replaced",

        "removed",

        "has been removed",

        "reset",

        "password generated",

        "password has been sent",

        "password was sent",

        "unlocked",

        "unlock account",

        "access provided",

        "granted the",

        "enabled",

        "disabled",

        "recreated",

        "fixed by",

        "released",

        "session released",

        "restored",

        "activated",

        "completed successfully",

        "request completed",

        "changes complete",

        "changes were made",

        "changes are made",

        "set as requested",

        "printer has been created",

        "printers were recreated",

        "rebooted",

        "reconfigured",

        "configured a new",

        "synced",

        "provided the access",

        "has been granted"
    ]

    if any(
        pattern in s
        for pattern in actionable_patterns
    ):
        return "ACTIONABLE"

    # ======================================
    # 5. DEFAULT
    # ======================================

    return "NON_ACTIONABLE"


# ==========================================
# RECALCUL DE LA CLASSIFICATION
# ==========================================

knowledge_base["solution_type"] = (
    knowledge_base["Solution"]
    .apply(classify_solution)
)

print(
    knowledge_base["solution_type"]
    .value_counts()
)

solution_type
GENERIC           1401
NON_ACTIONABLE     599
ACTIONABLE         201
ADMINISTRATIVE      42
Name: count, dtype: int64


In [87]:
# ==========================================
# FILTRAGE FINAL DES CANDIDATS
# ==========================================

def is_non_informative_solution(solution):
    """
    Détecte les solutions qui n'apportent
    pas réellement une information exploitable.
    """

    if pd.isna(solution):
        return True

    text = str(solution).strip().lower()

    if not text:
        return True

    non_informative = {

        "resolved",
        "resolved.",

        "resolve",

        "solved",
        "solved.",

        "closed",
        "closed.",

        "done",
        "done.",

        "fixed",
        "fixed."
    }

    if text in non_informative:
        return True

    automatic_patterns = [

        "rule of 3",

        "rule of three",

        "set to status 'resolved'",

        "set to status resolved",

        "ticket was set to status"
    ]

    return any(
        pattern in text
        for pattern in automatic_patterns
    )


# ==========================================
# RECALCUL DES SOLUTIONS NON INFORMATIVES
# ==========================================

knowledge_base["non_informative"] = (
    knowledge_base["Solution"]
    .apply(is_non_informative_solution)
)


# ==========================================
# FILTRAGE DES CANDIDATS
# ==========================================

def filter_actionable_candidates(candidates):
    """
    Conserve uniquement les solutions
    réellement exploitables.

    Priorité :

    1. ACTIONABLE
    2. GENERIC informative

    Les solutions purement non informatives
    sont supprimées.
    """

    candidates = candidates.copy()

    # ======================================
    # 1. RETIRER NON INFORMATIVES
    # ======================================

    usable = candidates[
        ~candidates["non_informative"]
    ].copy()

    if usable.empty:
        return usable

    # ======================================
    # 2. ACTIONABLE
    # ======================================

    actionable = usable[
        usable["solution_type"]
        == "ACTIONABLE"
    ].copy()

    if not actionable.empty:
        return actionable

    # ======================================
    # 3. FALLBACK GENERIC INFORMATIVE
    # ======================================

    generic = usable[
        usable["solution_type"]
        == "GENERIC"
    ].copy()

    return generic

In [88]:
# ==========================================
# SCORING FINAL DES CLUSTERS
# ==========================================

def score_solution_clusters(
    actionable,
    clusters,
    solution_similarity_matrix
):
    """
    Calcule les caractéristiques
    de chaque cluster.
    """

    cluster_results = []

    if clusters.empty:
        return pd.DataFrame()

    for cluster_id in sorted(
        clusters["Cluster"].unique()
    ):

        # ==================================
        # Tickets du cluster
        # ==================================

        cluster_numbers = clusters.loc[
            clusters["Cluster"] == cluster_id,
            "Number"
        ].tolist()

        cluster_data = actionable[
            actionable["Number"].isin(
                cluster_numbers
            )
        ].copy()

        if cluster_data.empty:
            continue

        # ==================================
        # Similarité moyenne des tickets
        # ==================================

        ticket_similarity_mean = (
            cluster_data[
                "Ticket_Similarity"
            ].mean()
        )

        # ==================================
        # Meilleure similarité
        # ==================================

        ticket_similarity_max = (
            cluster_data[
                "Ticket_Similarity"
            ].max()
        )

        # ==================================
        # Cohérence des solutions
        # ==================================

        valid_tickets = (
            cluster_data["Number"]
            .tolist()
        )

        if len(valid_tickets) > 1:

            matrix = (
                solution_similarity_matrix.loc[
                    valid_tickets,
                    valid_tickets
                ].values
            )

            upper_values = matrix[
                np.triu_indices(
                    len(matrix),
                    k=1
                )
            ]

            solution_coherence = (
                float(upper_values.mean())
            )

        else:

            # ==================================
            # IMPORTANT
            #
            # Un seul ticket ne permet pas de
            # mesurer une cohérence historique.
            # ==================================

            solution_coherence = 0.50

        # ==================================
        # Résultat
        # ==================================

        cluster_results.append({

            "Cluster":
                cluster_id,

            "Number_of_Tickets":
                len(cluster_data),

            "Ticket_Similarity_Mean":
                ticket_similarity_mean,

            "Ticket_Similarity_Max":
                ticket_similarity_max,

            "Solution_Coherence":
                solution_coherence
        })

    return pd.DataFrame(
        cluster_results
    )

In [89]:
# ==========================================
# CLASSEMENT FINAL DES CLUSTERS
# ==========================================

def rank_solution_clusters(cluster_scores):
    """
    Classe les clusters selon :

    - pertinence des tickets
    - cohérence des solutions
    - support historique
    """

    if cluster_scores.empty:
        return cluster_scores.copy()

    scores = cluster_scores.copy()

    # ======================================
    # RELEVANCE SCORE
    # ======================================

    scores["Relevance_Score"] = (
        scores[
            "Ticket_Similarity_Mean"
        ]
    )

    # ======================================
    # COHERENCE SCORE
    # ======================================

    scores["Coherence_Score"] = (
        scores[
            "Solution_Coherence"
        ]
    )

    # ======================================
    # SUPPORT SCORE
    #
    # IMPORTANT :
    #
    # Un cluster singleton ne doit pas être
    # considéré comme fortement supporté.
    # ======================================

    max_support = (
        scores[
            "Number_of_Tickets"
        ].max()
    )

    scores["Support_Score"] = np.where(

        scores[
            "Number_of_Tickets"
        ] >= 2,

        np.log1p(
            scores[
                "Number_of_Tickets"
            ]
        )
        /
        np.log1p(max_support),

        0.0
    )

    # ======================================
    # SCORE FINAL
    # ======================================

    scores["Cluster_Final_Score"] = (

        0.50
        *
        scores["Relevance_Score"]

        +

        0.30
        *
        scores["Coherence_Score"]

        +

        0.20
        *
        scores["Support_Score"]
    )

    # ======================================
    # TRI
    # ======================================

    return (

        scores

        .sort_values(
            "Cluster_Final_Score",
            ascending=False
        )

        .reset_index(
            drop=True
        )
    )